# ViFinQA Qwen 2.5 14B grounded critic V2

This dedicated Kaggle run consumes only a hash-bound grounded-critic packet input. It does not bootstrap benchmark assets, retrieve tables, select values, execute an answer, repair OCR, or promote provenance. Every result remains `machine_provisional`.

In [ ]:
from pathlib import Path
import json
import os
import hashlib
import subprocess
import sys
import tarfile

REPO_DIR = Path('/kaggle/working/AI_guru')
SOURCE_ARCHIVE_NAME = 'ai_guru_grounded_critic_source_v2.bundle'
SOURCE_MANIFEST_NAME = 'ai_guru_grounded_critic_source_v2.manifest.json'
SOURCE_PROTOCOL = 'kaggle_grounded_critic_source_bundle_v1'
source_archives = sorted(Path('/kaggle/input').rglob(SOURCE_ARCHIVE_NAME))
source_manifests = sorted(Path('/kaggle/input').rglob(SOURCE_MANIFEST_NAME))
if len(source_archives) != 1 or len(source_manifests) != 1:
    raise FileNotFoundError('Attach exactly one private minimal critic source archive and manifest.')
SOURCE_ARCHIVE, SOURCE_MANIFEST = source_archives[0], source_manifests[0]
if SOURCE_ARCHIVE.parent != SOURCE_MANIFEST.parent:
    raise RuntimeError('Critic source archive and manifest must originate from the same Kaggle input.')
if REPO_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite existing working source directory: {REPO_DIR}')
source_manifest = json.loads(SOURCE_MANIFEST.read_text(encoding='utf-8'))
if source_manifest.get('protocol') != SOURCE_PROTOCOL:
    raise ValueError('Critic source bundle has unsupported protocol.')
source_archive = ((source_manifest.get('outputs') or {}).get('archive') or {})
if source_archive.get('file_name') != SOURCE_ARCHIVE.name:
    raise ValueError('Critic source bundle archive name is invalid.')
source_archive_sha = hashlib.sha256(SOURCE_ARCHIVE.read_bytes()).hexdigest()
if source_archive_sha != source_archive.get('sha256'):
    raise ValueError('Critic source bundle archive SHA-256 mismatch.')
source_contract = source_manifest.get('source_contract') or {}
source_contract_keys = ('contains_raw_reports', 'contains_labels', 'contains_research_artifacts', 'contains_credentials', 'eligible_for_evidence', 'eligible_for_training', 'eligible_for_submission', 'eligible_for_promotion')
if any(source_contract.get(key) is not False for key in source_contract_keys):
    raise ValueError('Critic source bundle violates its isolation contract.')
source_identity = source_manifest.get('source_bundle') or {}
identity_base = {key: value for key, value in source_identity.items() if key != 'source_tree_sha256'}
canonical_identity = json.dumps(identity_base, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')
if source_identity.get('protocol') != SOURCE_PROTOCOL or source_identity.get('source_tree_sha256') != hashlib.sha256(canonical_identity).hexdigest():
    raise ValueError('Critic source bundle source identity is invalid.')
source_files = source_identity.get('files')
if not isinstance(source_files, list) or not source_files:
    raise ValueError('Critic source bundle has no declared source files.')
expected_members = [entry.get('path') for entry in source_files] + ['SOURCE_BUNDLE.json']
if any(not isinstance(entry.get('path'), str) or Path(entry['path']).as_posix() != entry['path'] or not isinstance(entry.get('sha256'), str) or len(entry['sha256']) != 64 or Path(entry['path']).is_absolute() or '..' in Path(entry['path']).parts for entry in source_files) or len(set(expected_members)) != len(expected_members):
    raise ValueError('Critic source bundle has invalid source paths or SHA-256 values.')
with tarfile.open(SOURCE_ARCHIVE, mode='r:*') as archive:
    members = archive.getmembers()
    if [member.name for member in members] != expected_members or not all(member.isfile() for member in members):
        raise ValueError('Critic source bundle archive members differ from source identity.')
    embedded_identity = archive.extractfile('SOURCE_BUNDLE.json')
    if embedded_identity is None or json.loads(embedded_identity.read()) != source_identity:
        raise ValueError('Critic source bundle embedded identity differs from manifest.')
    for entry in source_files:
        member = archive.extractfile(entry['path'])
        if member is None or hashlib.sha256(member.read()).hexdigest() != entry['sha256']:
            raise ValueError('Critic source bundle member SHA-256 mismatch.')
    REPO_DIR.mkdir(parents=True)
    for member in members:
        source = archive.extractfile(member)
        if source is None:
            raise ValueError('Critic source bundle member cannot be read.')
        destination = REPO_DIR / member.name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(source.read())
for entry in source_files:
    if hashlib.sha256((REPO_DIR / entry['path']).read_bytes()).hexdigest() != entry['sha256']:
        raise ValueError('Extracted critic source bundle member SHA-256 mismatch.')
SOURCE_PROVENANCE = {'source_tree_sha256': source_identity['source_tree_sha256'], 'git_revision': source_identity.get('git_revision'), 'archive_sha256': source_archive_sha}
print('Verified immutable source tree:', SOURCE_PROVENANCE['source_tree_sha256'])
required = [
    REPO_DIR / 'scripts/run_qwen_grounded_critic_v2.py',
    REPO_DIR / 'scripts/build_grounded_critic_results_manifest_v2.py',
    REPO_DIR / 'src/finance_query/grounded_critic_qwen_v2.py',
]
if any(not path.is_file() for path in required):
    raise RuntimeError('The attached source snapshot is incomplete for Qwen critic V2.')
print('Source revision:', SOURCE_PROVENANCE.get('git_revision', 'unavailable'))
print('Packet input is independently hash-bound and may be a newer research revision.')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torch==2.12.1', 'torchvision==0.27.1', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers==4.48.3', 'accelerate>=1.0', 'bitsandbytes>=0.45'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator, then restart the session.')
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if gpu_memory_gib < 14:
    raise RuntimeError(f'Qwen2.5-14B 4-bit requires at least 14 GiB VRAM; found {gpu_memory_gib:.1f} GiB.')
print('Qwen GPU:', torch.cuda.get_device_name(0), f'{gpu_memory_gib:.1f} GiB', 'torch=', torch.__version__)


In [ ]:
GPU_CRITIC_RESULTS = Path('/kaggle/working/vifinqa_grounded_critic_v2')
GPU_CRITIC_RESULTS.mkdir(parents=True, exist_ok=True)
PACKET_NAME = 'grounded_critic_packets_v2.jsonl'
PACKET_MANIFEST_NAME = 'grounded_critic_packets_v2.manifest.json'
packet_dirs = [
    path.parent for path in Path('/kaggle/input').rglob(PACKET_NAME)
    if (path.parent / PACKET_MANIFEST_NAME).is_file()
]
if len(packet_dirs) != 1:
    raise RuntimeError(f'Expected exactly one V2 packet input directory; found {len(packet_dirs)}.')
PACKET_DIR = packet_dirs[0]
PACKETS = PACKET_DIR / PACKET_NAME
PACKET_MANIFEST = PACKET_DIR / PACKET_MANIFEST_NAME
packet_manifest = json.loads(PACKET_MANIFEST.read_text(encoding='utf-8'))
packet_sha = hashlib.sha256(PACKETS.read_bytes()).hexdigest()
if packet_sha != ((packet_manifest.get('outputs') or {}).get('packets') or {}).get('sha256'):
    raise ValueError('Critic packet SHA-256 differs from attached manifest.')
print('Grounded critic packets:', PACKETS, 'sha256=', packet_sha)
QWEN_CRITIC_OUTPUT = GPU_CRITIC_RESULTS / 'qwen14_grounded_critic_results_v2.jsonl'
QWEN_CRITIC_RUNTIME = GPU_CRITIC_RESULTS / 'qwen14_grounded_critic_runtime_v2.json'
critic_env = os.environ.copy()
critic_env.update({'CUDA_VISIBLE_DEVICES': '0', 'WANDB_DISABLED': 'true', 'WANDB_MODE': 'disabled'})
subprocess.run([
    sys.executable, 'scripts/run_qwen_grounded_critic_v2.py',
    '--packets', str(PACKETS), '--packets-manifest', str(PACKET_MANIFEST),
    '--output', str(QWEN_CRITIC_OUTPUT), '--runtime-output', str(QWEN_CRITIC_RUNTIME),
    '--model', 'Qwen/Qwen2.5-14B-Instruct', '--max-new-tokens', '768',
], cwd=REPO_DIR, env=critic_env, check=True)
runtime = json.loads(QWEN_CRITIC_RUNTIME.read_text(encoding='utf-8'))
QWEN_CRITIC_MANIFEST = GPU_CRITIC_RESULTS / 'qwen14_grounded_critic_results_v2.manifest.json'
subprocess.run([
    sys.executable, 'scripts/build_grounded_critic_results_manifest_v2.py',
    '--packets', str(PACKETS), '--packets-manifest', str(PACKET_MANIFEST),
    '--results', str(QWEN_CRITIC_OUTPUT), '--run-mode', 'qwen14b_4bit',
    '--runtime', runtime['runtime'], '--gpu', runtime['gpu'],
    '--model-revision', str(runtime['model_revision']),
    '--source-bundle-manifest', str(SOURCE_MANIFEST), '--source-bundle-archive', str(SOURCE_ARCHIVE),
    '--output', str(QWEN_CRITIC_MANIFEST),
], cwd=REPO_DIR, env=critic_env, check=True)
result_manifest = json.loads(QWEN_CRITIC_MANIFEST.read_text(encoding='utf-8'))
assert result_manifest['inputs']['source_bundle']['source_tree_sha256'] == SOURCE_PROVENANCE['source_tree_sha256']
print(QWEN_CRITIC_MANIFEST.read_text(encoding='utf-8'))
print('Research-only: all output remains machine_provisional; no repair, evidence, training, or submission promotion was created.')